## Multivariate Time Series

Multivariate time series adalah rangkaian data di mana setiap titik data terdiri dari beberapa variabel yang diamati atau diukur secara bersamaan dalam interval waktu yang berurutan. Berbeda dengan univariate time series, kita tidak hanya melihat satu variabel pada satu waktu tertentu, tetapi sejumlah variabel yang diamati secara bersamaan. Ini memungkinkan kita untuk memahami bagaimana variabel-variabel tersebut berinteraksi satu sama lain seiring waktu.

In [1]:
import pandas as pd
import tensorflow as tf

In [5]:
df = pd.read_csv(
    'https://drive.google.com/uc?id=1AZRfFoyekqSYpri5183RmJjciRGz_ood',
    sep=',',
    index_col='datetime',
    header=0,
    parse_dates=['datetime']
)
df

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...
2007-02-14 17:19:00,0.636,0.140,241.16,2.6,0.0,0.0,0.0
2007-02-14 17:20:00,0.552,0.000,240.46,2.2,0.0,0.0,0.0
2007-02-14 17:21:00,0.538,0.000,239.74,2.2,0.0,0.0,0.0


In [6]:
def normalize_series(data, min, max):
    data = data - min
    data = data / max
    return data

data = df.values
data = normalize_series(data, data.min(axis=0), data.max(axis=0))

In [7]:
N_FEATURES = len(df.columns)

In [8]:
SPLIT_TIME = int(len(data) * 0.5)
x_train = data[:SPLIT_TIME]
x_valid = data[SPLIT_TIME:]

In [9]:
def windowed_dataset(series, batch_size, n_past=24, n_future=24, shift=1):
    ds = tf.data.Dataset.from_tensor_slices(series)
    ds = ds.window(size=n_past + n_future, shift=shift, drop_remainder=True)
    ds = ds.flat_map(lambda w: w.batch(n_past + n_future))
    ds = ds.map(lambda w: (w[:n_past], w[n_past:]))
    return ds.batch(batch_size).prefetch(1)

In [10]:
BATCH_SIZE = 32
N_PAST = 24
N_FUTURE = 24
SHIFT = 1

train_set = windowed_dataset(series=x_train, batch_size=BATCH_SIZE, n_past=N_PAST, n_future=N_FUTURE, shift=SHIFT)
valid_set = windowed_dataset(series=x_valid, batch_size=BATCH_SIZE, n_past=N_PAST, n_future=N_FUTURE, shift=SHIFT)


In [11]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(64, input_shape=(N_PAST, N_FUTURE)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(N_FEATURES)
])

/Users/yutaaa/Developments/daily-climate-series/.venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
class myCallback(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs={}):
            if (logs.get('mae') < 0.055 and logs.get('val_mae') < 0.055):
                self.model.stop_training = True
 
callbacks = myCallback()

In [23]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3) 
model.compile(loss='mae',
                  optimizer= optimizer,
                  metrics=["mae"])

model.fit(train_set,
          validation_data=(valid_set),
          epochs=100,
          callbacks=callbacks,
          verbose=1
    )

Epoch 1/100


/Users/yutaaa/Developments/daily-climate-series/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


   1345/Unknown 3s 2ms/step - loss: 0.0658 - mae: 0.0658

/Users/yutaaa/Developments/daily-climate-series/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1349/1349 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0657 - mae: 0.0657 - val_loss: 0.0519 - val_mae: 0.0519
Epoch 2/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - loss: 0.0529 - mae: 0.0529 - val_loss: 0.0509 - val_mae: 0.0509


In [25]:
train_pred = model.predict(train_set)
train_pred[0][0]


1349/1349 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


array([ 0.5138889 ,  0.1881669 , -0.00316535,  0.49549928, -0.02281209,
        0.00695526,  0.7136453 ], dtype=float32)